In [ ]:
# ======================================================================
# 0. Instalación de Dependencias (Ejecutar solo la primera vez)
# ===============================================================================

%pip install -q pandas==2.1.0 mysql-connector-python==8.1.0 python-dotenv==1.0.0 requests==2.31.0 ipywidgets==8.1.1

print("Dependencias instaladas y listas para usar.")

In [ ]:
# ==============================================================================
# 1 Celda de configuración: carga de .env, conección a MySQL.
# ==============================================================================

import os
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv

load_dotenv()

# Devuelve un objeto de conexión a la base de datos MySQL.
def conectar_bd():
    try:
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=int(os.getenv("DB_PORT")),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD") 
        )
        if conexion.is_connected(): return conexion

    except Error as e:
        print(f"Error: {e}")

    except Exception as e:
        print(f"Error: {e}")

    return None

# Prueba de conexión a la base de datos.
def probar_conexion():
    conexion = conectar_bd()

    if not conexion:
        print("Error de conexión: No se pudo conectar a la base de datos.")
        return

    # Hace una consulta sencilla.
    try:
        with conexion.cursor() as cursor:
            cursor.execute("SELECT COUNT(*) FROM LIBRO;")
            total_libros = cursor.fetchone()[0]
            print(f"Conexión exitosa.\nLibros en BD: {total_libros}")

    except Error as e:
        print(f"Error: {e}")

    except Exception as e:
        print(f"Error: {e}")

    finally:
        # Si la conexión sigue abierta, la cerramos.
        if conexion.is_connected():
            conexion.close()

if __name__ == "__main__":
    probar_conexion()

In [ ]:
# ==============================================================================
# 2 System Prompt: contexto, esquema, reglas de traducción de preguntas a SQL y Few-Shots de ejemplo.
# ==============================================================================

SYSTEM_PROMPT = """
Eres un asistente experto en bases de datos relacionales y traducción de lenguaje natural a SQL.
Tu única tarea es transformar preguntas en español a consultas SQL válidas para un motor MySQL 8+.
Solo vas a responder preguntas en base al esquema. No tienes permiso de insertar ni modificar información.

CONTEXTO DEL ESQUEMA DE LA BASE DE DATOS:
A continuación se detalla la estructura exacta de las tablas de la biblioteca "BiblioIA":

1. TABLA: AUTOR
   - id_autor (INT, PK, AUTO_INCREMENT)
   - nombre (VARCHAR)
   - apellido (VARCHAR)
   - nacionalidad (VARCHAR)

2. TABLA: GENERO
   - id_genero (INT, PK, AUTO_INCREMENT)
   - nombre (VARCHAR)
   - descripcion (TEXT)

3. TABLA: LIBRO
   - isbn (VARCHAR, PK)
   - titulo (VARCHAR)
   - anio_publicacion (INT)
   - stock_total (INT)
   - stock_disponible (INT)

4. TABLA: LIBRO_AUTOR (Relación N:M)
   - isbn (VARCHAR, FK -> LIBRO.isbn)
   - id_autor (INT, FK -> AUTOR.id_autor)
   PRIMARY KEY (isbn, id_autor)

5. TABLA: LIBRO_GENERO (Relación N:M)
   - isbn (VARCHAR, FK -> LIBRO.isbn)
   - id_genero (INT, FK -> GENERO.id_genero)
   PRIMARY KEY (isbn, id_genero)

6. TABLA: EJEMPLAR
   - id_ejemplar (INT, PK, AUTO_INCREMENT)
   - isbn (VARCHAR, FK -> LIBRO.isbn)
   - nro_ejemplar (INT)
   - estado_fisico (VARCHAR)

7. TABLA: SOCIO
   - id_socio (INT, PK, AUTO_INCREMENT)
   - dni (VARCHAR, UNIQUE)
   - nombre (VARCHAR)
   - apellido (VARCHAR)
   - email (VARCHAR, UNIQUE)
   - fecha_alta (DATE)
   - estado (VARCHAR)

8. TABLA: PRESTAMO
   - id_prestamo (INT, PK, AUTO_INCREMENT)
   - id_socio (INT, FK -> SOCIO.id_socio)
   - id_ejemplar (INT, FK -> EJEMPLAR.id_ejemplar)
   - fecha_prestamo (DATE)
   - fecha_vencimiento (DATE)
   - fecha_devolucion (DATE, NULL si no se devolvió)
   - estado (VARCHAR)

9. TABLA: SANCION
   - id_sancion (INT, PK, AUTO_INCREMENT)
   - id_socio (INT, FK -> SOCIO.id_socio)
   - tipo (VARCHAR)
   - fecha_inicio (DATE)
   - fecha_fin (DATE)
   - motivo (TEXT)

REGLAS CRÍTICAS DE SALIDA:
- Responde ÚNICAMENTE con el código SQL puro y válido.
- NO incluyas introducciones, explicaciones, comentarios, ni bloques de formato markdown (NO uses ```sql ... ```).
- Si la pregunta no se puede responder con el esquema provisto, devuelve únicamente el texto: ERROR_ESQUEMA.

EJEMPLOS DE REFERENCIA (FEW-SHOT PROMPTING):

Pregunta: ¿Cuáles son los 5 libros más prestados este año?
Respuesta: SELECT l.titulo, COUNT(p.id_prestamo) AS total_prestamos FROM PRESTAMO p JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar JOIN LIBRO l ON e.isbn = l.isbn WHERE YEAR(p.fecha_prestamo) = YEAR(CURDATE()) GROUP BY l.isbn, l.titulo ORDER BY total_prestamos DESC LIMIT 5;

Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
Respuesta: SELECT DISTINCT s.id_socio, s.nombre, s.apellido, s.dni FROM SOCIO s JOIN PRESTAMO p ON s.id_socio = p.id_socio WHERE p.estado = 'VENCIDO' OR (p.fecha_vencimiento < CURDATE() AND p.fecha_devolucion IS NULL);
"""


print("System Prompt listo.")

In [ ]:
# ==============================================================================
# 3 Función text_to_sql: pasa lenguaje natural a la API y retorna SQL.
# ==============================================================================

import os
import requests

# Genera una consulta SQL a partir de una pregunta en lenguaje natural usando la API de Gemini.
def text_to_sql(pregunta, system_prompt):
    api_key = os.getenv("LLM_API_KEY")
    if not api_key:
        print("Error: No se encontró la variable LLM_API_KEY en el archivo .env")
        return None

    url_gemini = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    
    payload = {
        "contents": [{
            "parts": [
                {"text": system_prompt},
                {"text": f"Pregunta del usuario: {pregunta}\nSQL: "}
                # Acá usamos un truquito que se conoce como "output forcing". Separamos la pregunta del usuario de la respuesta
                # del agente usando un salto de linea (\n), y establecemos un prefijo hardcodeado (SQL: ) en el punto en el que
                # el modelo empieza a generar. Esto reduce las posibilidades de que el modelo se vuelva loco y empiece a generar
                # texto innecesario y vaya al grano con la consulta SQL.
            ]
        }],
        "generationConfig": {
            "temperature": 0.0,
            "maxOutputTokens": 300
        }
    }

    try:
        respuesta = requests.post(url_gemini, json=payload, timeout=30)
        respuesta.raise_for_status() # Salta directamente al except si Google devuelve un código HTTP de error.
        
        resultado_json = respuesta.json() # Convierte la respuesta en un objeto JSON navegable.

        # Extrae el primer candidato a respuesta -> Extrae los contenidos del candidato -> Extrae la primer parte
        # del contenido, que tiene el bloque de texto -> Extrae dicho texto -> Elimina los vestigios del formato .md, de existir.
        sql_generado = resultado_json['candidates'][0]['content']['parts'][0]['text'].replace("```sql", "").replace("```", "").strip()
        # Y, por último, agrega el ';' en caso de que el agente se lo haya comido.
        if not sql_generado.endswith(";"): sql_generado += ";"  
        # Checkeamos que el SQL generado sea una consulta SELECT y no se modifique ni elimine la información de la BD.
        if not sql_generado.startswith("SELECT"): raise Exception("La pregunta no fue o no pudo ser respondida con una consulta SELECT")
        
        return sql_generado
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
# ==============================================================================
# 4 Función ejecutar_consulta: ejecuta en Mysql y retorna DataFrame.
# ==============================================================================

import pandas as pd
from mysql.connector import Error

# Ejecuta una consulta en la base de datos y devuelve un DataFrame.
def ejecutar_consulta(sql):

    # Crea una conección a la base de datos.
    conexion = conectar_bd()
    if not conexion:
        return None
    
    try:
        # Envuelve el cursor en un bloque contexto, garantiza que se cierre y se liberen los recursos.
        # dictionary=True altera la estructura retornada para que devuelva diccionarios en lugar de tuplas.
        with conexion.cursor(dictionary=True) as cursor:
            cursor.execute(sql)
            registros = cursor.fetchall()
            # Convierte los diccionarios devueltos en una tabla. Cambia el formato del nombre de las columnas:
            # id_libro -> Id Libro, por ej.
            return pd.DataFrame(registros).rename(columns=lambda x: x.replace('_', ' ').title(), inplace=True)
        
    except Error as e:
        print(f"Error: {e}")
        return None
    
    except Exception as e:
        print(f"Error: {e}")
        return None
    
    finally:
        if conexion and conexion.is_connected():
            conexion.close()

In [ ]:
# ==============================================================================
# 5 Función agente_responder: Orquestador principal
# ==============================================================================

from IPython.display import display

# Traduce lenguaje natural en resultados.
def agente_responder(pregunta, system_prompt):

    sql_generado = text_to_sql(pregunta, system_prompt)

    if not sql_generado:
        print("Error: No se pudo generar la consulta SQL.")
        return None, None

    if "ERROR_ESQUEMA" in sql_generado:
        print("Consulta no soportada, excede al esquema de la base de datos.")
        return None, None

    # Principalmente por motivos de debugging
    print(f"SQL Generado:\n{sql_generado}\n")

    # 2. Tu función entra en acción de forma segura
    df_resultados = ejecutar_consulta(sql_generado)

    if df_resultados is None:
        print("Fallo en la ejecución de la consulta SQL.")
    elif df_resultados.empty:
        print("La consulta se ejecutó correctamente, pero no hay datos para mostrar...")
    else:
        print("Resultado:")
        display(df_resultados)
    
    return df_resultados, sql_generado

In [ ]:
# ==============================================================================
# 6 Demo interactiva: widget ipywidgets para preguntas libres.
# ==============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

txt_pregunta = widgets.Text(
    value='',
    placeholder='Escribí acá tu consulta...',
    description='Consulta:',
    layout=widgets.Layout(width='70%')
)
btn_consultar = widgets.Button(description='Preguntar', button_style='primary', icon='search')
area_salida = widgets.Output()

def procesar_click(b):
    with area_salida:
        clear_output(wait=True) 
        pregunta = txt_pregunta.value.strip()
        if pregunta:
            print(f"Procesando: '{pregunta}'...")
            agente_responder(pregunta, SYSTEM_PROMPT)

btn_consultar.on_click(procesar_click)
txt_pregunta.continuous_update = False
txt_pregunta.observe(lambda c: procesar_click(None), 'value')

if not globals().get('MODO_TEST', False):
    print("PANEL INTERACTIVO BIBLIOIA")
    display(widgets.HBox([txt_pregunta, btn_consultar]), area_salida)

In [ ]:
# ==============================================================================
# 7 Módulo de recomendaciones con IA Generativa (Texto narrativo)
# ==============================================================================

from IPython.display import Markdown
# Nota: os, requests, ipywidgets, display y clear_output se heredan de celdas anteriores.


def generar_recomendacion_ia(prompt_recomendacion):
    api_key = os.getenv("LLM_API_KEY")
    if not api_key:
        print("Error: Clave API no configurada.")
        return None

    url_gemini = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    payload = {
        "contents": [{"parts": [{"text": prompt_recomendacion}]}],
        "generationConfig": {"temperature": 0.7, "maxOutputTokens": 400}
    }

    try:
        respuesta = requests.post(url_gemini, json=payload, timeout=15)
        respuesta.raise_for_status()

        # Misma limpieza de antes.
        return respuesta.json()['candidates'][0]['content']['parts'][0]['text']
    
    except Exception as e:
        print(f"Error: {e}")
    
    return None


def recomendar_para(id_socio):
    if not str(id_socio).isdigit():
        display(Markdown("Error: El ID de socio debe ser numérico."))
        return None
        
    id_socio_str = str(id_socio)

    sql_socio = f"SELECT nombre, apellido FROM SOCIO WHERE id_socio = {id_socio_str};"
    df_socio = ejecutar_consulta(sql_socio)
    
    if df_socio is None or df_socio.empty:
        display(Markdown(f"No se encontró socio con el ID {id_socio_str}."))
        return None

    nombre = df_socio.iloc[0]['Nombre']
    apellido = df_socio.iloc[0]['Apellido']
    display(Markdown(f"## Recomendaciones para {nombre} {apellido}"))

    sql_historial = f"""
        SELECT DISTINCT l.titulo, g.nombre AS genero
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO l ON e.isbn = l.isbn
        JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn
        JOIN GENERO g ON lg.id_genero = g.id_genero
        WHERE p.id_socio = {id_socio_str};
    """
    df_historial = ejecutar_consulta(sql_historial)
    texto_historial = ""
    texto_match = ""
    match_nombre = None
    df_libros_match = None

    if df_historial is not None and not df_historial.empty:
        display(Markdown("**📚 Tu historial de lectura:**"))
        display(df_historial)
        lista_historial = [f"- {r['Titulo']} (Género: {r['Genero']})" for _, r in df_historial.iterrows()]
        texto_historial = "\n".join(lista_historial)

        sql_match = f"""
            SELECT s.id_socio, s.nombre, s.apellido, COUNT(DISTINCT g.id_genero) as coincidencias
            FROM PRESTAMO p
            JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
            JOIN LIBRO_GENERO lg ON e.isbn = lg.isbn
            JOIN GENERO g ON lg.id_genero = g.id_genero
            JOIN SOCIO s ON p.id_socio = s.id_socio
            WHERE p.id_socio != {id_socio_str}
              AND g.id_genero IN (
                  SELECT DISTINCT lg2.id_genero
                  FROM PRESTAMO p2
                  JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar
                  JOIN LIBRO_GENERO lg2 ON e2.isbn = lg2.isbn
                  WHERE p2.id_socio = {id_socio_str}
              )
            GROUP BY s.id_socio, s.nombre, s.apellido
            ORDER BY coincidencias DESC LIMIT 1;
        """
        df_match = ejecutar_consulta(sql_match)
        
        if df_match is not None and not df_match.empty:
            id_match = df_match.iloc[0]['Id Socio']
            match_nombre = df_match.iloc[0]['Nombre']
            match_apellido = df_match.iloc[0]['Apellido']
            texto_match = f"Encontramos un Match literario con {match_nombre} {match_apellido}."

            sql_libros_match = f"""
                SELECT DISTINCT l.titulo
                FROM PRESTAMO p
                JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
                JOIN LIBRO l ON e.isbn = l.isbn
                WHERE p.id_socio = {id_match} AND l.isbn NOT IN (
                    SELECT e2.isbn FROM PRESTAMO p2 JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar WHERE p2.id_socio = {id_socio_str}
                ) AND l.stock_disponible > 0 LIMIT 5;
            """
            df_libros_match = ejecutar_consulta(sql_libros_match)
            if df_libros_match is not None and not df_libros_match.empty:
                texto_match += "\nLibros que leyó tu match: " + ", ".join(df_libros_match['Titulo'].tolist())
        else:
            texto_match = "No encontramos otro socio con gustos similares."
    else:
        display(Markdown("*No registras historial previo.*"))
        texto_historial = "El usuario es nuevo."
        texto_match = "Sin historial no hay match."

    sql_candidatos = f"""
        SELECT l.titulo
        FROM LIBRO l
        WHERE l.stock_disponible > 0
          AND l.isbn NOT IN (
              SELECT e.isbn FROM PRESTAMO p JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar WHERE p.id_socio = {id_socio_str}
          ) ORDER BY RAND() LIMIT 3;
    """
    df_candidatos = ejecutar_consulta(sql_candidatos)
    if df_candidatos is None or df_candidatos.empty:
        display(Markdown("⚠️ No hay libros disponibles para recomendar."))
        return None
        
    texto_candidatos = "\n".join([f"- {t}" for t in df_candidatos['Titulo']])

    prompt_recomendacion = f"""Eres un bibliotecario recomendando libros a {nombre}.
Historial: {texto_historial}
Recomendar: {texto_candidatos}
Match: {texto_match}
Habla coloquial, breve y sin markdown complejo. Al final cuenta el estado del match."""

    texto_ia = generar_recomendacion_ia(prompt_recomendacion)
    if texto_ia:
        display(Markdown(texto_ia))
    
    if df_libros_match is not None and not df_libros_match.empty:
        display(Markdown(f"### 📖 Libros de {match_nombre} que todavía no leíste"))
        display(df_libros_match)

# Widget UI Block
txt_socio = widgets.Text(value='1', description='ID Socio:')
btn_recomendar = widgets.Button(description='Recomendar', button_style='success')
area_recomendacion = widgets.Output()

def procesar_recomendacion(b):
    with area_recomendacion:
        clear_output(wait=True)
        recomendar_para(txt_socio.value)

btn_recomendar.on_click(procesar_recomendacion)

if not globals().get('MODO_TEST', False):
    display(widgets.HBox([txt_socio, btn_recomendar]), area_recomendacion)

In [ ]:
# ==============================================================================
# 8. Módulo de alertas de devolución inteligente
# ==============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# State management encapsulated in a dictionary to avoid 'global' side-effects
estado_alertas = {
    'df': None,
    'ultimo_indice': 0
}

def buscar_alertas(dias):
    sql_alertas = f"""
        SELECT s.id_socio, s.nombre, s.apellido, s.email, l.titulo, p.fecha_vencimiento,
               DATEDIFF(p.fecha_vencimiento, CURDATE()) AS dias_restantes
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO l ON e.isbn = l.isbn
        JOIN SOCIO s ON p.id_socio = s.id_socio
        WHERE p.estado = 'ACTIVO'
          AND p.fecha_devolucion IS NULL
          AND p.fecha_vencimiento BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL {dias} DAY)
        ORDER BY p.fecha_vencimiento ASC;
    """
    df = ejecutar_consulta(sql_alertas)
    
    if df is None or df.empty:
        print(f"✅ No hay préstamos que venzan en los próximos {dias} día(s).")
        estado_alertas['df'] = None
        btn_generar_mensajes.layout.display = 'none'
        return
        
    estado_alertas['df'] = df
    estado_alertas['ultimo_indice'] = 0
    
    # Keys matched to Cell 5 title-casing format
    display(df[['Nombre', 'Apellido', 'Email', 'Titulo', 'Fecha Vencimiento', 'Dias Restantes']])
    btn_generar_mensajes.layout.display = 'inline-flex'
    print("¿Generar mensajes?")

def generar_mensajes():
    df = estado_alertas['df']
    if df is None or df.empty: return
    
    desde = estado_alertas['ultimo_indice']
    
    # Grouping using Title Case keys
    socios = list(df.groupby(['Id Socio', 'Nombre', 'Apellido', 'Email']))
    hasta = min(desde + 3, len(socios))
    
    for i in range(desde, hasta):
        (id_socio, nom, ape, email), grp = socios[i]
        
        libros = "\n".join([f"- {r['Titulo']} (vence {r['Fecha Vencimiento']})" for _, r in grp.iterrows()])
        
        prompt = f"Eres el sistema automatizado de BiblioIA. Redacta un correo electrónico de recordatorio breve, profesional y muy cordial para el socio {nom} {ape} avisándole que los siguientes libros vencen pronto:\n{libros}\nDespídete cordialmente. No uses formato markdown complejo."
        
        # Utilizing the standardized API layer from Cell 8
        texto_ia = generar_recomendacion_ia(prompt)
        
        if texto_ia:
            display(Markdown(f"---\n### 📧 Para: {email} (Socio: {nom} {ape})\n" + texto_ia))
            
    estado_alertas['ultimo_indice'] = hasta
    
    if hasta < len(socios):
        btn_continuar.layout.display = 'inline-flex'
    else:
        btn_continuar.layout.display = 'none'
        display(Markdown("**✅ Todos los mensajes han sido generados.**"))

# Widget Configuration
txt_dias = widgets.BoundedIntText(value=10, min=1, max=30, description='Días:', layout=widgets.Layout(width='20%'))
btn_buscar = widgets.Button(description='Buscar alertas', button_style='warning')
btn_generar_mensajes = widgets.Button(description='Generar mensajes', button_style='success', layout=widgets.Layout(display='none'))
btn_continuar = widgets.Button(description='Generar más', button_style='info', layout=widgets.Layout(display='none'))
area_tabla = widgets.Output()
area_mensajes = widgets.Output()

def procesar_busqueda(b):
    with area_tabla:
        clear_output(wait=True)
        buscar_alertas(txt_dias.value)
    with area_mensajes:
        clear_output(wait=True)
        btn_continuar.layout.display = 'none'

def procesar_generacion(b):
    with area_mensajes:
        # No clear_output() here to allow progressive appending of emails
        generar_mensajes()
        btn_generar_mensajes.layout.display = 'none' # Hide initial generate button after first click

btn_buscar.on_click(procesar_busqueda)
btn_generar_mensajes.on_click(procesar_generacion)
btn_continuar.on_click(procesar_generacion)

if not globals().get('MODO_TEST', False):
    display(widgets.HBox([txt_dias, btn_buscar, btn_generar_mensajes, btn_continuar]))
    display(area_tabla)
    display(area_mensajes)

In [ ]:
# ==============================================================================
# 9. Dashboard Dinámico con IA (Texto a Gráfico)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROMPT_GRAFICO = """
Eres un asistente experto en bases de datos relacionales y traducción de lenguaje natural a SQL.
Tu única tarea es transformar preguntas en español a consultas SQL válidas para un motor MySQL 8+.
Solo vas a responder preguntas en base al esquema. No tienes permiso de insertar ni modificar información.

CONTEXTO DEL ESQUEMA DE LA BASE DE DATOS:
A continuación se detalla la estructura exacta de las tablas de la biblioteca "BiblioIA":

1. TABLA: AUTOR
   - id_autor (INT, PK, AUTO_INCREMENT)
   - nombre (VARCHAR)
   - apellido (VARCHAR)
   - nacionalidad (VARCHAR)

2. TABLA: GENERO
   - id_genero (INT, PK, AUTO_INCREMENT)
   - nombre (VARCHAR)
   - descripcion (TEXT)

3. TABLA: LIBRO
   - isbn (VARCHAR, PK)
   - titulo (VARCHAR)
   - anio_publicacion (INT)
   - stock_total (INT)
   - stock_disponible (INT)

4. TABLA: LIBRO_AUTOR (Relación N:M)
   - isbn (VARCHAR, FK -> LIBRO.isbn)
   - id_autor (INT, FK -> AUTOR.id_autor)
   PRIMARY KEY (isbn, id_autor)

5. TABLA: LIBRO_GENERO (Relación N:M)
   - isbn (VARCHAR, FK -> LIBRO.isbn)
   - id_genero (INT, FK -> GENERO.id_genero)
   PRIMARY KEY (isbn, id_genero)

6. TABLA: EJEMPLAR
   - id_ejemplar (INT, PK, AUTO_INCREMENT)
   - isbn (VARCHAR, FK -> LIBRO.isbn)
   - nro_ejemplar (INT)
   - estado_fisico (VARCHAR)

7. TABLA: SOCIO
   - id_socio (INT, PK, AUTO_INCREMENT)
   - dni (VARCHAR, UNIQUE)
   - nombre (VARCHAR)
   - apellido (VARCHAR)
   - email (VARCHAR, UNIQUE)
   - fecha_alta (DATE)
   - estado (VARCHAR)

8. TABLA: PRESTAMO
   - id_prestamo (INT, PK, AUTO_INCREMENT)
   - id_socio (INT, FK -> SOCIO.id_socio)
   - id_ejemplar (INT, FK -> EJEMPLAR.id_ejemplar)
   - fecha_prestamo (DATE)
   - fecha_vencimiento (DATE)
   - fecha_devolucion (DATE, NULL si no se devolvió)
   - estado (VARCHAR)

9. TABLA: SANCION
   - id_sancion (INT, PK, AUTO_INCREMENT)
   - id_socio (INT, FK -> SOCIO.id_socio)
   - tipo (VARCHAR)
   - fecha_inicio (DATE)
   - fecha_fin (DATE)
   - motivo (TEXT)

REGLAS CRÍTICAS DE SALIDA:

-El usuario pedirá una estadística. Debes devolver una consulta SQL que retorne EXACTAMENTE DOS COLUMNAS:
    1. Una categoría (texto).
    2. Un valor numérico (cantidad de registros, sumas, etc.).
- Responde ÚNICAMENTE con el código SQL puro y válido.
- NO incluyas introducciones, explicaciones, comentarios, ni bloques de formato markdown (NO uses ```sql ... ```).
- Si la pregunta no se puede responder con el esquema provisto, devuelve únicamente el texto: ERROR_ESQUEMA.
"""

txt_pregunta_grafico = widgets.Text(
    value='',
    placeholder='Ej: ¿Cuántos socios hay por estado?',
    description='Gráfico de:',
    layout=widgets.Layout(width='70%')
)
btn_generar_dinamico = widgets.Button(description='Dibujar con IA', button_style='success', icon='magic')
area_grafico_dinamico = widgets.Output()

def procesar_grafico_dinamico(b):
    with area_grafico_dinamico:
        clear_output(wait=True)
        pregunta = txt_pregunta_grafico.value.strip()
        if not pregunta: 
            return
        
        print(f"Analizando: '{pregunta}'...")
        
        # Integration with Cell 3 abstraction layer
        sql_generado = text_to_sql(pregunta, PROMPT_GRAFICO)
        
        if sql_generado:
            if "ERROR_ESQUEMA" in sql_generado:
                print("Consulta no soportada: La pregunta excede el esquema de la base de datos.")
                return
                
            df_datos = ejecutar_consulta(sql_generado)
            
            if df_datos is not None and not df_datos.empty:
                if len(df_datos.columns) >= 2:
                    col_x = df_datos.columns[0]
                    col_y = df_datos.columns[1]
                    
                    # Data type coercion to prevent Matplotlib rendering crashes
                    df_datos[col_y] = pd.to_numeric(df_datos[col_y], errors='coerce')
                    df_datos = df_datos.dropna(subset=[col_y])
                    
                    if df_datos.empty:
                        print("Error: La consulta no devolvió valores numéricos graficables en la segunda columna.")
                        return
                    
                    fig, ax = plt.subplots(figsize=(8, 5))
                    ax.bar(df_datos[col_x].astype(str), df_datos[col_y], color='#8e44ad')
                    
                    # Labels utilize pre-mutated keys from Cell 5 execution
                    ax.set_xlabel(col_x, fontweight='bold')
                    ax.set_ylabel(col_y, fontweight='bold')
                    ax.set_title(pregunta.title(), fontsize=14, fontweight='bold')
                    ax.grid(axis='y', linestyle='--', alpha=0.7)
                    plt.xticks(rotation=15)
                    
                    plt.tight_layout()
                    plt.show()
                else:
                    print("La IA no devolvió el formato esperado de 2 columnas.")
            else:
                print("No hay datos para graficar con esa consulta.")

btn_generar_dinamico.on_click(procesar_grafico_dinamico)

if not globals().get('MODO_TEST', False):
    display(widgets.HBox([txt_pregunta_grafico, btn_generar_dinamico]), area_grafico_dinamico)